In [1]:
import json
import csv
import os
import io # Import io for StringIO

# --- Function to make the text more readable ---
def make_text_more_readable(text_input):
    """
    Formats the raw play-by-play text for better readability.
    Adds paragraph breaks, bolding for key terms, and half markers.
    """
    if not text_input:
        return ""

    # Initial paragraph breaks based on common sentence endings
    formatted_text = text_input.replace(". ", ".\n\n")
    formatted_text = formatted_text.replace("! ", "!\n\n")
    formatted_text = formatted_text.replace("? ", "?\n\n")

    # Add bolder markers for half transitions
    formatted_text = formatted_text.replace(
        "At the end of the first half, the score is tied at 1-1 between the Home Team and the Away Team.",
        "**--- END OF FIRST HALF (Score: Home Team 1 - Away Team 1) ---**\n\n"
    )
    formatted_text = formatted_text.replace(
        "The second half is underway with the score tied at 1-1 between the Home Team and the Away Team.",
        "\n\n**--- START OF SECOND HALF ---**\n\n"
    )
    formatted_text = formatted_text.replace(
        "The second half has finished with both teams tied at one goal each.",
        "\n\n**--- END OF SECOND HALF (Score: Home Team 1 - Away Team 1) ---**\n\n"
    )
    formatted_text = formatted_text.replace(
        "The game finishes with a 1-1 draw between the Home Team and the Away Team.",
        "\n\n**--- FINAL WHISTLE (Final Score: Home Team 1 - Away Team 1 Draw) ---**\n\n"
    )

    # Bold important keywords for easy scanning
    bold_keywords = [
        "GOAL!", "PENALTY!", "YELLOW CARD!", "RED CARD!",
        "Home Team", "Away Team", "offside", "corner kick", "free kick",
        "scores", "fouls", "assists", "saved"
    ]
    for keyword in bold_keywords:
        # Use regex to find whole words to avoid partial matches
        import re
        formatted_text = re.sub(r'\b' + re.escape(keyword) + r'\b', r'**\g<0>**', formatted_text, flags=re.IGNORECASE)
        # Handle cases like "Team's" or "Team)"
        formatted_text = re.sub(r'\b(' + re.escape(keyword) + r')([\'\)\.])\b', r'**\g<1>*\g<2>**', formatted_text, flags=re.IGNORECASE)


    # Add a title at the beginning if not already there
    if not formatted_text.strip().startswith("### Match Commentary:"):
        formatted_text = "### Match Commentary: Home Team vs. Away Team\n\n" + formatted_text

    return formatted_text

# --- Main Script ---
# Load the JSON file
try:
    with open('train.json', 'r') as file:
        data = json.load(file)
except FileNotFoundError:
    print("Error: 'train.json' not found. Please ensure the file is in the same directory.")
    exit()
except json.JSONDecodeError:
    print("Error: Could not decode 'train.json'. Please check if it's a valid JSON file.")
    exit()

# Create a directory to store the files
output_dir = 'training_data'
os.makedirs(output_dir, exist_ok=True)

# Determine how many objects to process (max 100 or fewer if data has less)
num_objects_to_process = min(500, len(data))

# Process the specified number of objects
for i, obj in enumerate(data[:num_objects_to_process]):
    # --- Process 'table' parameter ---
    table_raw_data = obj.get('table', '')

    table_data_for_csv = []
    if isinstance(table_raw_data, str) and table_raw_data:
        # Replace custom newline tag with actual newline characters
        csv_string = table_raw_data.replace('<NEWLINE>', '\n')
        # Use io.StringIO to read the string as if it were a file
        # Then use csv.reader to parse the CSV string into rows
        reader = csv.reader(io.StringIO(csv_string))
        table_data_for_csv = list(reader) # Convert reader object to a list of lists
    elif isinstance(table_raw_data, list):
        # If it's already a list of lists, use it directly
        table_data_for_csv = table_raw_data

    csv_filename = f'sample_{i+1}_table.csv'
    csv_filepath = os.path.join(output_dir, csv_filename)
    try:
        with open(csv_filepath, 'w', newline='') as csv_file: # newline='' is recommended for csv.writer
            writer = csv.writer(csv_file)
            writer.writerows(table_data_for_csv)
    except IOError as e:
        print(f"Error writing CSV file '{csv_filepath}': {e}")
        continue # Skip to the next object

    # --- Process 'text' parameter ---
    raw_text_data = obj.get('text', '')

    # Apply the readability formatting
    formatted_text_data = make_text_more_readable(raw_text_data)

    text_filename = f'sample_{i+1}_commentary.md'
    text_filepath = os.path.join(output_dir, text_filename)
    try:
        with open(text_filepath, 'w') as text_file:
            text_file.write(formatted_text_data)
    except IOError as e:
        print(f"Error writing text file '{text_filepath}': {e}")
        continue # Skip to the next object

print(f"Successfully processed {num_objects_to_process} objects and saved files to '{output_dir}'.")
print("Check the 'training_data' directory for your formatted CSV and text files.")

Successfully processed 500 objects and saved files to 'training_data'.
Check the 'training_data' directory for your formatted CSV and text files.
